### Jacobi + Gauss-Seidel iterative solver for linear systems

Given a square linear system $A x = b$, the following code solves it iteratively by choosing automatically between the Jacobi method and the Gauss-Seidel method. The choice criterion is based the spectral radius

In [1]:
import numpy as np

In [2]:
def JGS(A, b, x0 = None, eps = 1e-06, max_iter = 1_000): # Jacobi/Gauss-Seidel solver that chooses method by the best spectral radius
    A = np.asarray(A, dtype = float, copy = True)
    b = np.asarray(b, dtype = float, copy = True)
    n = len(b)  # Number of equations or unknowns
    x = np.asarray(x0, dtype = float, copy = True) if x0 is not None else np.zeros(n) # initial guess or zero vector

    d = np.diag(A)  # diagonal entries of A
    if np.any(d == 0.0): raise ValueError("Zero element on the diagonal of A") # Error handling

    # Iteration matrices
    B_jacobi = np.eye(n) - A / d[:, None] # Jacobi iteration matrix: I - D^{-1} @ A
    B_gs = - np.linalg.solve(np.tril(A), np.triu(A, 1)) # GS iteration matrix: - (D + L)^{-1} @ U; tril(A) = D + L, triu(A , 1) = U
    
    # Computing spectral radius of iteration matrices
    rho_jacobi = np.max(np.abs(np.linalg.eigvals(B_jacobi)))
    rho_gs = np.max(np.abs(np.linalg.eigvals(B_gs)))

    if (rho_jacobi >= 1.0) and (rho_gs >= 1.0): # Neither method is guaranteed to converge
        print(f"Both methods are non-convergent: rho(Jacobi)={rho_jacobi:.6g}, rho(GS)={rho_gs:.6g}")
        return None

    use_gs = rho_gs <= rho_jacobi # Choose method with smaller/equal spectral radius
    print("Gauss-Seidel chosen" if use_gs else "Jacobi chosen")

    r = b - A @ x # initial residual r = b - A x
    if np.linalg.norm(r) < eps: return x  # If already converged

    if use_gs: # Gauss-Seidel branch
        for _ in range(max_iter): # Iterate up to max_iter sweeps
            for i in range(n): # Update each unknown in order
                delta = r[i] / d[i] # GS update for x_i using current residual
                x[i] += delta # Apply update to x_i
                r -= delta * A[:, i] # Update residual after changing x_i

            if np.linalg.norm(r) < eps: # Check convergence after each sweep
                return x
            if not np.all(np.isfinite(x)): # Stop if overflow/NaN appears
                break

    else: # Jacobi branch
        for _ in range(max_iter): # Iterate up to max_iter iterations
            delta = r / d # Simultaneous Jacobi update: D^{-1} @ r
            x += delta # Update all unknowns at once
            r -= A @ delta # Update residual: r = b - A x

            if np.linalg.norm(r) < eps: # Check convergence
                return x
            if not np.all(np.isfinite(x)): # Stop if overflow/NaN appears
                break

    print("Desired precision not reached. Consider to increase the maximum number of iterations or increase the tolerance")
    return x # Return best/last approximation if not converged

Let's run some examples

In [3]:
A = np.array([
    [3.0, 0.0, 4.0],
    [7.0, 4.0, 2.0],
    [-1.0, -1.0, -2.0]
    ])
b = np.array([7.0, 13.0, -4.0])

sol = JGS(A, b)
sol

Gauss-Seidel chosen


array([1.00000008, 0.99999989, 1.00000001])

In [4]:
# Error estimation
residual = b - A @ sol
np.abs(JGS(A, residual, residual, 1e-09))

Gauss-Seidel chosen


array([7.95504699e-08, 1.09381897e-07, 1.49157134e-08])

In [5]:
A = np.array([
    [-3.0, 3.0, - 6.0],
    [-4.0, 7.0, -8.0],
    [5.0, 7.0, -9.0]
    ])
b = np.array([-6.0, -5.0, 3.0])
x0 = np.array([0.0, 0.0, 0.0])

sol = JGS(A, b)
sol

Jacobi chosen


array([0.99999992, 1.00000006, 1.        ])

In [6]:
# Error estimation
residual = b - A @ sol
np.abs(JGS(A, residual, residual, 1e-09))

Jacobi chosen


array([8.37045795e-08, 6.26347728e-08, 2.72146218e-09])

In [7]:
A = np.array([
    [4.0, 1.0, 1.0],
    [2.0, -9.0, 0.0],
    [0.0, -8.0, -6.0]
    ])
b = np.array([6.0, -7.0, -14.0])
x0 = np.array([0.0, 0.0, 0.0])

sol = JGS(A, b)
sol

Gauss-Seidel chosen


array([1.00000006, 1.00000001, 0.99999998])

In [8]:
# Error estimation
residual = b - A @ sol
np.abs(JGS(A, residual, residual, 1e-09))

Gauss-Seidel chosen


array([5.87822228e-08, 1.30627164e-08, 1.74169552e-08])

In [9]:
A = np.array([
    [7.0, 6.0, 9.0],
    [4.0, 5.0, - 4.0],
    [-7.0, - 3.0, 8.0]
    ])
b = np.array([22.0, 5.0, - 2.0])
x0 = np.array([0.0, 0.0, 0.0])

sol = JGS(A, b)
sol

Jacobi chosen


array([1.00000008, 0.99999994, 0.99999998])

In [10]:
# Error estimation
residual = b - A @ sol
np.abs(JGS(A, residual, residual, 1e-09))

Jacobi chosen


array([8.26566642e-08, 5.52290843e-08, 2.47919113e-08])

In [11]:
A = np.array([
    [1, 2, 2],
    [2, 1, 2],
    [2, 2, 1]
    ])
b = np.array([5, 5, 5])
x0 = np.array([0.0, 0.0, 0.0])

sol = JGS(A, b)
sol

Both methods are non-convergent: rho(Jacobi)=4, rho(GS)=2.82843
